# Insurance Charge Predictor — evaluation walkthrough

A supervised **regression** project predicting the benchmark `charges` target. All reported values below are read from the trained project artifacts. Amounts are USD medical charges, not live premiums.


In [1]:
from pathlib import Path
import sys, json
import pandas as pd
PROJECT = Path.cwd()
assert (PROJECT / 'data/raw.csv').exists(), 'Open this notebook from the project folder.'
sys.path.insert(0, str(PROJECT / 'src'))
from predict import predict, load_bundle
print('Project:', PROJECT.name)


Project: insurance_charge_predictor


## Inspect and audit the data

Preserve the original table. Verify actual missing and duplicate counts before claiming to clean anything.


In [2]:
raw = pd.read_csv(PROJECT / 'data/raw.csv')
clean = pd.read_csv(PROJECT / 'data/clean.csv')
print('Raw rows:', len(raw))
print('Exact duplicate rows:', int(raw.duplicated().sum()))
print('Missing cells:', int(raw.isna().sum().sum()))
print('Clean rows:', len(clean))
print(clean.head().to_string(index=False))


Raw rows: 1338
Exact duplicate rows: 1
Missing cells: 0
Clean rows: 1337
 age    sex    bmi  children smoker    region     charges
  19 female 27.900         0    yes southwest 16884.92400
  18   male 33.770         1     no southeast  1725.55230
  28   male 33.000         3     no southeast  4449.46200
  33   male 22.705         0     no northwest 21984.47061
  32   male 28.880         0     no northwest  3866.85520


## Separate inputs from the answer

The target `charges` must be excluded from input X. Categorical values are encoded inside the fitted pipeline. Preprocessing is learned only within each training partition.


In [3]:
from schema import FEATURES
X = clean[FEATURES]
y = clean['charges']
print('Inputs:', list(X.columns))
print('Target:', y.name)
manifest = pd.read_csv(PROJECT / 'data/split_manifest.csv')
print(manifest['split'].value_counts().to_string())


Inputs: ['age', 'sex', 'bmi', 'children', 'smoker', 'region']
Target: charges
split
train    1069
test      268


## Compare algorithms without touching the test set

These are five-fold cross-validation results from training records only. The winning model has the lowest CV MAE. The mean baseline provides a simple reference.


In [4]:
comparison = pd.read_csv(PROJECT / 'reports/model_comparison.csv')
print(comparison.round(4).to_string(index=False))


            model  cv_mae_usd  cv_mae_std  cv_rmse_usd   cv_r2
Gradient boosting   2097.3715    277.2313    4842.5903  0.8344
    Random forest   2778.8437    349.8565    4839.6915  0.8328
 Ridge regression   4398.3850    358.2314    6297.0095  0.7225
    Mean baseline   9155.4571    814.5616   12098.1776 -0.0154


## Evaluate the selected model on the held-out test records

R² is a regression score, not a classification accuracy percentage. MAE is an average error, not an individual prediction interval.


In [5]:
metrics = json.loads((PROJECT / 'reports/metrics.json').read_text())
print(json.dumps(metrics, indent=2))
print('Fitted steps:', list(load_bundle()['pipeline'].named_steps))


{
  "selected_model": "Gradient boosting",
  "selection_rule": "Lowest mean 5-fold training CV MAE",
  "test_mae_usd": 1352.6541893323372,
  "test_rmse_usd": 3297.496253342969,
  "test_r2": 0.9245711772266488,
  "train_rows": 1069,
  "test_rows": 268,
  "seed": 42,
  "currency": "USD"
}
Fitted steps: ['prepare', 'model']


![Held-out predictions](reports/figures/actual_vs_predicted.png)

![Residuals](reports/figures/residuals.png)


## Predict two illustrative profiles

Only smoking status differs. This illustrates model associations, not a causal experiment.


In [6]:
profiles = pd.DataFrame([
    {'age':45, 'sex':'male', 'bmi':31.5, 'children':2, 'smoker':'no', 'region':'southeast'},
    {'age':45, 'sex':'male', 'bmi':31.5, 'children':2, 'smoker':'yes', 'region':'southeast'},
])
print(predict(profiles).to_string(index=False))


 age  sex  bmi  children smoker    region  predicted_charges_usd
  45 male 31.5         2     no southeast                8934.37
  45 male 31.5         2    yes southeast               40856.24


## Reproduce the experiment

From the project terminal, run:

```bash
python src/train.py
python src/verify.py
python run.py
```

Open http://127.0.0.1:8502. If training changes the model, restart the app to reload its cached pipeline. Source attribution is in `data/SOURCE.md`; limitations and the full methodology are in `reports/PROJECT_REPORT.md`.
